# 02 — Data Quality & Validation

## 1. Introduction and objectives

Phase 1A helped us understand what Olist contains; Phase 1B checks whether its values and relationships are sufficiently consistent for later analysis. Exploration describes the data, while validation tests expectations and reports exceptions without correcting them. We validate orders, order items, customers, payments, reviews, products, sellers, and the product-category translation table when available. They provide order progress and commerce context for CallVerse scenarios involving tracking, delays, returns, refunds, and cancellations. Olist does not record actual customer contacts.

## 2. Imports and dataset loading

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR, "exists:", DATA_DIR.exists())
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory missing: {DATA_DIR}. Put Olist CSVs in data/raw/olist/."
    )

selected_files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
datasets = {}
for dataset_name, file_name in selected_files.items():
    file_path = DATA_DIR / file_name
    if file_path.exists():
        datasets[dataset_name] = pd.read_csv(file_path)
    else:
        print("Dataset unavailable; related checks will be skipped:", file_path)
orders_df = datasets.get("orders")
quality_findings = []

Project root: C:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse
Data directory: C:\Users\user\OneDrive - ESPRIT\Bureau\CallVerse\data\raw\olist exists: True


## 3. Dataset overview

This inventory records dimensions, columns, pandas-inferred types, and memory use. Date columns are parsed explicitly later.

In [2]:
overview_rows = []
for dataset_name, dataset in datasets.items():
    overview_rows.append(
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "columns": len(dataset.columns),
            "column_names": ", ".join(dataset.columns.astype(str)),
            "data_types": "; ".join(
                f"{column}: {dtype}" for column, dtype in dataset.dtypes.items()
            ),
            "memory_mb": round(
                dataset.memory_usage(index=True, deep=True).sum() / 1024**2, 2
            ),
        }
    )
display(pd.DataFrame(overview_rows))

,dataset,rows,columns,column_names,data_types,memory_mb
0,orders,99441,8,"order_id, customer_id, order_status, order_pur...",order_id: object; customer_id: object; order_s...,52.94
1,order_items,112650,7,"order_id, order_item_id, product_id, seller_id...",order_id: object; order_item_id: int64; produc...,35.99
2,customers,99441,5,"customer_id, customer_unique_id, customer_zip_...",customer_id: object; customer_unique_id: objec...,26.59
3,payments,103886,5,"order_id, payment_sequential, payment_type, pa...",order_id: object; payment_sequential: int64; p...,16.23
4,reviews,99224,7,"review_id, order_id, review_score, review_comm...",review_id: object; order_id: object; review_sc...,39.12
5,products,32951,9,"product_id, product_category_name, product_nam...",product_id: object; product_category_name: obj...,6.30
6,sellers,3095,4,"seller_id, seller_zip_code_prefix, seller_city...",seller_id: object; seller_zip_code_prefix: int...,0.59
7,category_translation,71,2,"product_category_name, product_category_name_e...",product_category_name: object; product_categor...,0.01


## 4. Missing values analysis

For every column, count missing values and their percentage of rows, sorted by percentage. Missing delivery times may be legitimate for canceled or in-progress orders. This phase reports missingness without filling values.

In [3]:
callverse_columns = {
    "orders": [
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["order_id", "product_id", "seller_id"],
    "customers": ["customer_id"],
    "payments": ["order_id", "payment_type", "payment_value"],
    "reviews": ["order_id", "review_score"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}
missing_summaries = {}
for dataset_name, dataset in datasets.items():
    missing_count = dataset.isna().sum()
    summary = pd.DataFrame(
        {
            "column": missing_count.index,
            "missing_values": missing_count.values,
            "missing_percent": missing_count.values / len(dataset) * 100
            if len(dataset)
            else 0,
        }
    )
    summary = summary.sort_values("missing_percent", ascending=False)
    missing_summaries[dataset_name] = summary
    for _, row in summary[summary.missing_values > 0].iterrows():
        quality_findings.append(
            {
                "Check": "Missing values",
                "Dataset": dataset_name,
                "Issue investigated": row.column,
                "Number of affected records": int(row.missing_values),
                "Interpretation": f"{row.missing_percent:.2f}% missing; assess in context.",
            }
        )
    if summary.missing_values.sum() == 0:
        quality_findings.append(
            {
                "Check": "Missing values",
                "Dataset": dataset_name,
                "Issue investigated": "All columns",
                "Number of affected records": 0,
                "Interpretation": "No missing values.",
            }
        )
    print("\nMissing values:", dataset_name)
    display(summary)
    relevant = summary[
        (summary.missing_values > 0)
        & summary.column.isin(callverse_columns.get(dataset_name, []))
    ]
    if not relevant.empty:
        print("CallVerse-relevant missing columns:")
        display(relevant)


Missing values: orders


,column,missing_values,missing_percent
6,order_delivered_customer_date,2965,2.981668
5,order_delivered_carrier_date,1783,1.793023
4,order_approved_at,160,0.160899
0,order_id,0,0.000000
1,customer_id,0,0.000000
2,order_status,0,0.000000
3,order_purchase_timestamp,0,0.000000
7,order_estimated_delivery_date,0,0.000000


CallVerse-relevant missing columns:


,column,missing_values,missing_percent
6,order_delivered_customer_date,2965,2.981668
5,order_delivered_carrier_date,1783,1.793023
4,order_approved_at,160,0.160899



Missing values: order_items


...

,column,missing_values,missing_percent
1,product_category_name,610,1.851234
2,product_name_lenght,610,1.851234
3,product_description_lenght,610,1.851234
4,product_photos_qty,610,1.851234
5,product_weight_g,2,0.006070
6,product_length_cm,2,0.006070
7,product_height_cm,2,0.006070
8,product_width_cm,2,0.006070
0,product_id,0,0.000000



Missing values: sellers


,column,missing_values,missing_percent
0,seller_id,0,0.0
1,seller_zip_code_prefix,0,0.0
2,seller_city,0,0.0
3,seller_state,0,0.0



Missing values: category_translation


,column,missing_values,missing_percent
0,product_category_name,0,0.0
1,product_category_name_english,0,0.0


## 5. Duplicate analysis

Completely identical rows differ from repeated identifiers. Check expected record keys only: order and customer IDs, product and seller IDs, review IDs, and the order plus item number composite. Order IDs can repeat in items, payments, and reviews. In Olist, `customer_id` is created for each order, so it is expected to be unique in both orders and customers; the person behind an order is `customer_unique_id`, which can legitimately repeat. For reviews we also test the composite key (`review_id`, `order_id`). No rows are deleted.

In [4]:
duplicate_findings = []
key_expectations = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "order_items": [["order_id", "order_item_id"]],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "reviews": ["review_id", ["review_id", "order_id"]],
    "category_translation": ["product_category_name"],
}
for name, dataset in datasets.items():
    exact_count = int(dataset.duplicated().sum())
    duplicate_findings.append(
        {
            "dataset": name,
            "check": "Completely duplicated rows",
            "columns": "all columns",
            "affected_records": exact_count,
        }
    )
    quality_findings.append(
        {
            "Check": "Duplicates",
            "Dataset": name,
            "Issue investigated": "Completely duplicated rows",
            "Number of affected records": exact_count,
            "Interpretation": "Exact duplicate count; rows retained.",
        }
    )
    for key in key_expectations.get(name, []):
        columns = key if isinstance(key, list) else [key]
        if all(column in dataset for column in columns):
            repeated_count = int(dataset.duplicated(subset=columns, keep=False).sum())
            duplicate_findings.append(
                {
                    "dataset": name,
                    "check": "Repeated expected identifier",
                    "columns": ", ".join(columns),
                    "affected_records": repeated_count,
                }
            )
            quality_findings.append(
                {
                    "Check": "Identifier uniqueness",
                    "Dataset": name,
                    "Issue investigated": ", ".join(columns),
                    "Number of affected records": repeated_count,
                    "Interpretation": "Rows with repeated expected record key; inspect before joining.",
                }
            )
    for column in ["order_id", "customer_id", "customer_unique_id"]:
        if column in dataset and column not in key_expectations.get(name, []):
            count = int(dataset[column].duplicated(keep=False).sum())
            if count:
                duplicate_findings.append(
                    {
                        "dataset": name,
                        "check": "Repeated identifier (may be legitimate)",
                        "columns": column,
                        "affected_records": count,
                    }
                )
                quality_findings.append(
                    {
                        "Check": "Repeated identifier",
                        "Dataset": name,
                        "Issue investigated": column,
                        "Number of affected records": count,
                        "Interpretation": "May be legitimate: table granularity or repeat customers.",
                    }
                )
duplicate_summary = pd.DataFrame(duplicate_findings)
display(duplicate_summary)

,dataset,check,columns,affected_records
0,orders,Completely duplicated rows,all columns,0
1,orders,Repeated expected identifier,order_id,0
2,order_items,Completely duplicated rows,all columns,0
3,order_items,Repeated expected identifier,"order_id, order_item_id",0
4,order_items,Repeated identifier (may be legitimate),order_id,23787
5,customers,Completely duplicated rows,all columns,0
6,customers,Repeated expected identifier,customer_id,0
7,customers,Repeated identifier (may be legitimate),customer_unique_id,6342
8,payments,Completely duplicated rows,all columns,0
9,payments,Repeated identifier (may be legitimate),order_id,7407


### Repeated review identifiers

`review_id` is not unique. A repeated identifier is not automatically an error, so we check what the repeated rows have in common: whether they point to different orders, whether the review content is identical, and whether the orders belong to the same person (`customer_unique_id`). We also count orders that have more than one review row.

In [5]:
reviews_df = datasets.get("reviews")
customers_df = datasets.get("customers")
if reviews_df is not None and orders_df is not None and customers_df is not None:
    repeated_reviews = reviews_df[
        reviews_df.duplicated(subset=["review_id"], keep=False)
    ].copy()

    # Find the person behind each order: order_id -> customer_id -> customer_unique_id
    customer_by_order = orders_df.set_index("order_id")["customer_id"]
    person_by_customer = customers_df.set_index("customer_id")["customer_unique_id"]
    repeated_reviews["customer_unique_id"] = (
        repeated_reviews["order_id"].map(customer_by_order).map(person_by_customer)
    )

    # Empty texts are replaced by a marker only for this comparison, so two empty values count as equal
    comparison = repeated_reviews.fillna(
        {"review_comment_title": "<empty>", "review_comment_message": "<empty>"}
    )
    content_columns = [
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
    ]
    review_groups = comparison.groupby("review_id")
    copies = review_groups.size()
    distinct_orders = review_groups["order_id"].nunique()
    distinct_people = review_groups["customer_unique_id"].nunique()
    identical_content = (review_groups[content_columns].nunique() == 1).all(axis=1)

    display(
        pd.DataFrame(
            {
                "check": [
                    "Repeated review_id values",
                    "...every copy points to a different order",
                    "...all copies have identical review content",
                    "...all orders belong to the same person",
                ],
                "count": [
                    len(copies),
                    int((distinct_orders == copies).sum()),
                    int(identical_content.sum()),
                    int((distinct_people == 1).sum()),
                ],
            }
        )
    )
    example_columns = [
        "review_id",
        "order_id",
        "customer_unique_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp",
    ]
    display(
        repeated_reviews.sort_values(["review_id", "order_id"])[example_columns].head(6)
    )

    reviews_per_order = reviews_df["order_id"].value_counts()
    orders_with_several_reviews = int((reviews_per_order > 1).sum())
    print("Orders with more than one review row:", orders_with_several_reviews)

    quality_findings.append(
        {
            "Check": "Identifier investigation",
            "Dataset": "reviews",
            "Issue investigated": "Repeated review_id with identical content and same person",
            "Number of affected records": int(
                (identical_content & (distinct_people == 1)).sum()
            ),
            "Interpretation": "Count of review_id values; suggests one review linked to several orders. (review_id, order_id) is unique.",
        }
    )
    quality_findings.append(
        {
            "Check": "Identifier investigation",
            "Dataset": "reviews",
            "Issue investigated": "Orders with more than one review row",
            "Number of affected records": orders_with_several_reviews,
            "Interpretation": "Which review is final must be decided before order-level use.",
        }
    )
else:
    print("Review identifier checks unavailable: reviews, orders or customers missing.")

,check,count
0,Repeated review_id values,789
1,...every copy points to a different order,789
2,...all copies have identical review content,789
3,...all orders belong to the same person,789


,review_id,order_id,customer_unique_id,review_score,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,f30856ad31d3e74253a3f4ccef670648,1,2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,f30856ad31d3e74253a3f4ccef670648,1,2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,91ffaf5eb7f4bd6a48b07f3546fb7a99,5,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,91ffaf5eb7f4bd6a48b07f3546fb7a99,5,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,ba16ab3721ffd6b57fc87a7a4f6a4fcf,1,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,ba16ab3721ffd6b57fc87a7a4f6a4fcf,1,2018-03-07 00:00:00,2018-03-08 03:00:53


Orders with more than one review row: 547


If every repeated `review_id` has identical content and all its orders belong to the same person, the most plausible explanation is that **one review was attached to several orders of the same customer**. This is a hypothesis supported by the pattern, not confirmed by Olist documentation. The records are not removed; `(review_id, order_id)` is the appropriate record key for the reviews table.

## 6. Identifier and referential integrity validation

Compare each available child identifier with the parent identifier set. Repeated child keys are allowed. Missing keys are counted separately from unmatched non-missing references.

In [6]:
relationship_specs = [
    ("Orders to customers", "orders", "customer_id", "customers", "customer_id"),
    ("Items to orders", "order_items", "order_id", "orders", "order_id"),
    ("Items to products", "order_items", "product_id", "products", "product_id"),
    ("Items to sellers", "order_items", "seller_id", "sellers", "seller_id"),
    ("Payments to orders", "payments", "order_id", "orders", "order_id"),
    ("Reviews to orders", "reviews", "order_id", "orders", "order_id"),
]
relationship_results = []
for label, child_name, child_key, parent_name, parent_key in relationship_specs:
    child, parent = datasets.get(child_name), datasets.get(parent_name)
    if (
        child is None
        or parent is None
        or child_key not in child
        or parent_key not in parent
    ):
        relationship_results.append(
            {"relationship": label, "status": "table or key unavailable"}
        )
        continue
    values = child[child_key]
    nonmissing = values.dropna()
    unmatched = ~nonmissing.isin(set(parent[parent_key].dropna().unique()))
    count = int(unmatched.sum())
    relationship_results.append(
        {
            "relationship": label,
            "status": "checked",
            "child_rows": len(child),
            "distinct_child_ids": nonmissing.nunique(),
            "distinct_parent_ids": parent[parent_key].nunique(),
            "missing_child_ids": int(values.isna().sum()),
            "unmatched_references": count,
        }
    )
    quality_findings.append(
        {
            "Check": "Referential integrity",
            "Dataset": child_name,
            "Issue investigated": label,
            "Number of affected records": count,
            "Interpretation": "Unmatched non-missing references; review if non-zero.",
        }
    )
display(pd.DataFrame(relationship_results))

,relationship,status,child_rows,distinct_child_ids,distinct_parent_ids,missing_child_ids,unmatched_references
0,Orders to customers,checked,99441,99441,99441,0,0
1,Items to orders,checked,112650,98666,99441,0,0
2,Items to products,checked,112650,32951,32951,0,0
3,Items to sellers,checked,112650,3095,3095,0,0
4,Payments to orders,checked,103886,99440,99441,0,0
5,Reviews to orders,checked,99224,98673,99441,0,0


## 7. Timestamp consistency validation

Parse the available order milestones, count non-empty values that fail parsing, and compare chronology only when both values exist. Suspicious order is worth investigation but is not proof of error. Missing events may match an order's state. Later predictive features must use only information available at simulated contact time, not future delivery outcomes.

`order_estimated_delivery_date` is a calendar date stored at midnight. Comparing it directly with the delivery *timestamp* would count an order delivered later on the estimated day as late, so lateness is compared by calendar date. The timestamp comparison is shown for reference only.

In [7]:
timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
timestamp_results = []
if orders_df is not None:
    orders_df = (
        orders_df.copy()
    )  # parsed columns go on a copy; datasets["orders"] stays as loaded
    for column in timestamp_columns:
        if column in orders_df:
            parsed = column + "_parsed"
            orders_df[parsed] = pd.to_datetime(orders_df[column], errors="coerce")
            invalid = int((orders_df[column].notna() & orders_df[parsed].isna()).sum())
            timestamp_results.append(
                {
                    "check": "Unparseable non-missing timestamps",
                    "comparison": column,
                    "comparable_records": int(orders_df[column].notna().sum()),
                    "suspicious_records": invalid,
                }
            )
            quality_findings.append(
                {
                    "Check": "Timestamp parsing",
                    "Dataset": "orders",
                    "Issue investigated": column,
                    "Number of affected records": invalid,
                    "Interpretation": "Non-empty values that did not parse.",
                }
            )
    pairs = [
        ("order_purchase_timestamp", "order_approved_at"),
        ("order_approved_at", "order_delivered_carrier_date"),
        ("order_delivered_carrier_date", "order_delivered_customer_date"),
        ("order_purchase_timestamp", "order_delivered_customer_date"),
    ]
    for earlier, later in pairs:
        a, b = earlier + "_parsed", later + "_parsed"
        if a in orders_df and b in orders_df:
            both = orders_df[a].notna() & orders_df[b].notna()
            count = int((both & (orders_df[a] > orders_df[b])).sum())
            label = earlier + " after " + later
            timestamp_results.append(
                {
                    "check": "Chronological order",
                    "comparison": label,
                    "comparable_records": int(both.sum()),
                    "suspicious_records": count,
                }
            )
            quality_findings.append(
                {
                    "Check": "Timestamp consistency",
                    "Dataset": "orders",
                    "Issue investigated": label,
                    "Number of affected records": count,
                    "Interpretation": "Compared where both times exist; investigate, not a confirmed error.",
                }
            )

    actual = orders_df["order_delivered_customer_date_parsed"]
    estimate = orders_df["order_estimated_delivery_date_parsed"]
    purchase = orders_df["order_purchase_timestamp_parsed"]

    # The estimate should be a pure calendar date (time 00:00:00)
    estimate_with_time = int(
        (estimate.notna() & (estimate != estimate.dt.normalize())).sum()
    )
    timestamp_results.append(
        {
            "check": "Estimate format",
            "comparison": "Estimated dates with a time other than 00:00:00",
            "comparable_records": int(estimate.notna().sum()),
            "suspicious_records": estimate_with_time,
        }
    )

    with_purchase = actual.notna() & purchase.notna()
    negative_duration = with_purchase & (actual < purchase)
    timestamp_results.append(
        {
            "check": "Delivery comparison",
            "comparison": "Negative purchase-to-delivery duration",
            "comparable_records": int(with_purchase.sum()),
            "suspicious_records": int(negative_duration.sum()),
        }
    )
    quality_findings.append(
        {
            "Check": "Timestamp consistency",
            "Dataset": "orders",
            "Issue investigated": "Negative purchase-to-delivery duration",
            "Number of affected records": int(negative_duration.sum()),
            "Interpretation": "Delivery before purchase would be impossible; investigate if non-zero.",
        }
    )

    with_estimate = actual.notna() & estimate.notna()
    late_by_date = with_estimate & (actual.dt.normalize() > estimate)
    late_by_timestamp = with_estimate & (actual > estimate)
    timestamp_results.append(
        {
            "check": "Delivery comparison",
            "comparison": "Delivery date later than estimated date",
            "comparable_records": int(with_estimate.sum()),
            "suspicious_records": int(late_by_date.sum()),
        }
    )
    timestamp_results.append(
        {
            "check": "Delivery comparison",
            "comparison": "Delivery timestamp later than estimate (reference only)",
            "comparable_records": int(with_estimate.sum()),
            "suspicious_records": int(late_by_timestamp.sum()),
        }
    )
    quality_findings.append(
        {
            "Check": "Delivery performance",
            "Dataset": "orders",
            "Issue investigated": "Delivery date later than estimated date",
            "Number of affected records": int(late_by_date.sum()),
            "Interpretation": "Lateness outcome, not erroneous data; compared by calendar date.",
        }
    )
display(pd.DataFrame(timestamp_results))

,check,comparison,comparable_records,suspicious_records
0,Unparseable non-missing timestamps,order_purchase_timestamp,99441,0
1,Unparseable non-missing timestamps,order_approved_at,99281,0
2,Unparseable non-missing timestamps,order_delivered_carrier_date,97658,0
3,Unparseable non-missing timestamps,order_delivered_customer_date,96476,0
4,Unparseable non-missing timestamps,order_estimated_delivery_date,99441,0
5,Chronological order,order_purchase_timestamp after order_approved_at,99281,0
6,Chronological order,order_approved_at after order_delivered_carrie...,97644,1359
7,Chronological order,order_delivered_carrier_date after order_deliv...,96475,23
8,Chronological order,order_purchase_timestamp after order_delivered...,96476,0
9,Estimate format,Estimated dates with a time other than 00:00:00,99441,0


### Review timestamps

Reviews have a creation date (calendar date, stored at midnight) and an answer timestamp. We check that both parse, that no review is answered before it was created, and that no review is created before the purchase date of its order. Purchase timestamps are compared at calendar-date level for the same reason as delivery estimates.

In [8]:
reviews_df = datasets.get("reviews")
review_timestamp_results = []
if reviews_df is not None:
    review_dates = reviews_df[["review_id", "order_id"]].copy()
    for column in ["review_creation_date", "review_answer_timestamp"]:
        review_dates[column] = pd.to_datetime(reviews_df[column], errors="coerce")
        invalid = int((reviews_df[column].notna() & review_dates[column].isna()).sum())
        review_timestamp_results.append(
            {
                "check": "Unparseable non-missing timestamps",
                "comparison": column,
                "comparable_records": int(reviews_df[column].notna().sum()),
                "suspicious_records": invalid,
            }
        )
        quality_findings.append(
            {
                "Check": "Timestamp parsing",
                "Dataset": "reviews",
                "Issue investigated": column,
                "Number of affected records": invalid,
                "Interpretation": "Non-empty values that did not parse.",
            }
        )

    both = (
        review_dates["review_creation_date"].notna()
        & review_dates["review_answer_timestamp"].notna()
    )
    answered_before_created = both & (
        review_dates["review_answer_timestamp"] < review_dates["review_creation_date"]
    )
    review_timestamp_results.append(
        {
            "check": "Chronological order",
            "comparison": "Answer before creation",
            "comparable_records": int(both.sum()),
            "suspicious_records": int(answered_before_created.sum()),
        }
    )
    quality_findings.append(
        {
            "Check": "Timestamp consistency",
            "Dataset": "reviews",
            "Issue investigated": "Answer before creation",
            "Number of affected records": int(answered_before_created.sum()),
            "Interpretation": "Investigate if non-zero.",
        }
    )

    if orders_df is not None:
        purchase_by_order = orders_df.set_index("order_id")[
            "order_purchase_timestamp_parsed"
        ]
        purchase_date = review_dates["order_id"].map(purchase_by_order).dt.normalize()
        both = review_dates["review_creation_date"].notna() & purchase_date.notna()
        created_before_purchase = both & (
            review_dates["review_creation_date"] < purchase_date
        )
        review_timestamp_results.append(
            {
                "check": "Cross-table chronology",
                "comparison": "Review created before purchase date",
                "comparable_records": int(both.sum()),
                "suspicious_records": int(created_before_purchase.sum()),
            }
        )
        repeated_id = reviews_df.duplicated(subset=["review_id"], keep=False)
        print(
            "Reviews created before purchase date that also have a repeated review_id:",
            int((created_before_purchase & repeated_id).sum()),
        )
        quality_findings.append(
            {
                "Check": "Timestamp consistency",
                "Dataset": "reviews",
                "Issue investigated": "Review created before purchase date",
                "Number of affected records": int(created_before_purchase.sum()),
                "Interpretation": "Investigate; may relate to review_id values attached to several orders.",
            }
        )
    display(pd.DataFrame(review_timestamp_results))
else:
    print("Review timestamp checks unavailable: reviews missing.")

Reviews created before purchase date that also have a repeated review_id: 40


,check,comparison,comparable_records,suspicious_records
0,Unparseable non-missing timestamps,review_creation_date,99224,0
1,Unparseable non-missing timestamps,review_answer_timestamp,99224,0
2,Chronological order,Answer before creation,99224,0
3,Cross-table chronology,Review created before purchase date,99224,64


## 8. Order status and business consistency

Show statuses actually present, counts, and percentages. Delivery timestamp patterns are diagnostic: missing dates can be legitimate, and unusual combinations should be investigated rather than treated as absolute violations.

In [9]:
if orders_df is not None and "order_status" in orders_df:
    counts = orders_df.order_status.value_counts(dropna=False)
    display(
        pd.DataFrame(
            {
                "orders": counts,
                "percent": counts / len(orders_df) * 100 if len(orders_df) else 0,
            }
        )
    )
    print("Statuses observed:", orders_df.order_status.dropna().unique().tolist())
    delivery_columns = [
        column
        for column in [
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
        if column in orders_df
    ]
    display(
        orders_df.groupby("order_status", dropna=False)[delivery_columns].agg(
            lambda values: int(values.notna().sum())
        )
    )
    if (
        "delivered" in orders_df.order_status.dropna().unique()
        and "order_delivered_customer_date" in orders_df
    ):
        count = int(
            (
                orders_df.order_status.eq("delivered")
                & orders_df.order_delivered_customer_date.isna()
            ).sum()
        )
        quality_findings.append(
            {
                "Check": "Status and delivery timestamps",
                "Dataset": "orders",
                "Issue investigated": "Delivered without customer delivery timestamp",
                "Number of affected records": count,
                "Interpretation": "Diagnostic count, interpret with other order fields.",
            }
        )
    if (
        "canceled" in orders_df.order_status.dropna().unique()
        and "order_delivered_customer_date" in orders_df
    ):
        count = int(
            (
                orders_df.order_status.eq("canceled")
                & orders_df.order_delivered_customer_date.notna()
            ).sum()
        )
        quality_findings.append(
            {
                "Check": "Status and delivery timestamps",
                "Dataset": "orders",
                "Issue investigated": "Canceled with customer delivery timestamp",
                "Number of affected records": count,
                "Interpretation": "Diagnostic count; not by itself proof of error.",
            }
        )
else:
    print("Order status checks unavailable: orders or order_status missing.")

,orders,percent
order_status,,
delivered,96478,97.020344
shipped,1107,1.113223
canceled,625,0.628513
unavailable,609,0.612423
invoiced,314,0.315765
processing,301,0.302692
created,5,0.005028
approved,2,0.002011


Statuses observed: ['delivered', 'invoiced', 'shipped', 'processing', 'unavailable', 'canceled', 'created', 'approved']


,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,
approved,0,0,2
canceled,75,6,625
created,0,0,5
delivered,96476,96470,96478
invoiced,0,0,314
processing,0,0,301
shipped,1107,0,1107
unavailable,0,0,609


### Payment values

Payment rows should have a defined payment type, a positive value, and at least one instalment. Exceptions are counted, not changed; a zero-value voucher, for example, may be legitimate.

In [10]:
payments_df = datasets.get("payments")
if payments_df is not None:
    display(payments_df["payment_type"].value_counts().to_frame("payments"))
    payment_checks = [
        ("Payment type 'not_defined'", payments_df["payment_type"] == "not_defined"),
        ("Payment value zero or negative", payments_df["payment_value"] <= 0),
        ("Zero instalments", payments_df["payment_installments"] == 0),
    ]
    payment_results = []
    for label, mask in payment_checks:
        payment_results.append(
            {"check": label, "affected_payment_rows": int(mask.sum())}
        )
        quality_findings.append(
            {
                "Check": "Payment values",
                "Dataset": "payments",
                "Issue investigated": label,
                "Number of affected records": int(mask.sum()),
                "Interpretation": "Unusual value; investigate before payment features.",
            }
        )
    display(pd.DataFrame(payment_results))
else:
    print("Payment checks unavailable: payments missing.")

,payments
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


,check,affected_payment_rows
0,Payment type 'not_defined',3
1,Payment value zero or negative,9
2,Zero instalments,2


## 9. Cross-table consistency checks

Compare distinct order coverage and count orders without item, payment, or review rows. Then compare each order's payment total with its item prices plus freight, and check that every product category has an English translation. Different tables have different granularity and lifecycle coverage; unusual counts are observations requiring investigation.

In [11]:
coverage = []
if orders_df is not None and "order_id" in orders_df:
    order_ids = set(orders_df.order_id.dropna().unique())
    coverage.append(
        {
            "dataset": "orders",
            "rows": len(orders_df),
            "distinct order IDs": len(order_ids),
        }
    )
    for name in ["order_items", "payments", "reviews"]:
        dataset = datasets.get(name)
        if dataset is not None and "order_id" in dataset:
            child_ids = set(dataset.order_id.dropna().unique())
            absent = len(order_ids - child_ids)
            coverage.append(
                {
                    "dataset": name,
                    "rows": len(dataset),
                    "distinct order IDs": len(child_ids),
                    "orders without matching rows": absent,
                }
            )
            if name == "reviews":
                interpretation = (
                    "Reviews are optional; absence is expected for some orders."
                )
            else:
                interpretation = "Coverage observation; investigate table scope before calling it a defect."
            quality_findings.append(
                {
                    "Check": "Order coverage",
                    "Dataset": "orders and " + name,
                    "Issue investigated": "Orders without associated " + name,
                    "Number of affected records": absent,
                    "Interpretation": interpretation,
                }
            )
display(pd.DataFrame(coverage))

,dataset,rows,distinct order IDs,orders without matching rows
0,orders,99441,99441,NaN
1,order_items,112650,98666,775.0
2,payments,103886,99440,1.0
3,reviews,99224,98673,768.0


In [12]:
order_items_df = datasets.get("order_items")
payments_df = datasets.get("payments")
if order_items_df is not None and payments_df is not None:
    # Totals are computed only for this check
    item_value = order_items_df["price"] + order_items_df["freight_value"]
    items_total = item_value.groupby(order_items_df["order_id"]).sum()
    payments_total = payments_df.groupby("order_id")["payment_value"].sum()
    totals = pd.DataFrame(
        {"items_plus_freight": items_total, "payments": payments_total}
    ).dropna()
    totals["difference"] = totals["payments"] - totals["items_plus_freight"]
    differs = totals["difference"].abs() > 1.0  # tolerance of 1 BRL for rounding
    print("Orders with both items and payments:", len(totals))
    print(
        "Orders whose payments differ from items + freight by more than 1 BRL:",
        int(differs.sum()),
    )
    display(
        totals.loc[differs, "difference"]
        .describe()
        .round(2)
        .to_frame("payments_minus_items")
    )
    quality_findings.append(
        {
            "Check": "Cross-table values",
            "Dataset": "payments and order_items",
            "Issue investigated": "Payment total differs from items + freight by > 1 BRL",
            "Number of affected records": int(differs.sum()),
            "Interpretation": "May reflect instalment interest or vouchers; investigate.",
        }
    )

products_df = datasets.get("products")
translation_df = datasets.get("category_translation")
if products_df is not None and translation_df is not None:
    product_categories = set(products_df["product_category_name"].dropna())
    untranslated = sorted(
        product_categories - set(translation_df["product_category_name"])
    )
    affected_products = int(
        products_df["product_category_name"].isin(untranslated).sum()
    )
    print("Product categories without an English translation:", untranslated)
    print("Products in those categories:", affected_products)
    quality_findings.append(
        {
            "Check": "Cross-table coverage",
            "Dataset": "products and category_translation",
            "Issue investigated": "Products whose category has no translation",
            "Number of affected records": affected_products,
            "Interpretation": "Translation mapping incomplete for "
            + str(len(untranslated))
            + " categories.",
        }
    )

Orders with both items and payments: 98665
Orders whose payments differ from items + freight by more than 1 BRL: 249


,payments_minus_items
count,249.00
mean,11.51
std,19.35
min,-51.62
25%,3.29
50%,6.98
75%,14.12
max,182.81


Product categories without an English translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Products in those categories: 13


## 10. Data quality summary

The consolidated table uses counts calculated above and has no predetermined expected counts. Detailed missing-value and duplicate summaries remain in their earlier sections.

In [13]:
quality_summary = pd.DataFrame(
    quality_findings,
    columns=[
        "Check",
        "Dataset",
        "Issue investigated",
        "Number of affected records",
        "Interpretation",
    ],
)
with pd.option_context("display.max_rows", None):  # show every row of the summary
    display(quality_summary)

,Check,Dataset,Issue investigated,Number of affected records,Interpretation
0,Missing values,orders,order_delivered_customer_date,2965,2.98% missing; assess in context.
1,Missing values,orders,order_delivered_carrier_date,1783,1.79% missing; assess in context.
2,Missing values,orders,order_approved_at,160,0.16% missing; assess in context.
3,Missing values,order_items,All columns,0,No missing values.
4,Missing values,customers,All columns,0,No missing values.
5,Missing values,payments,All columns,0,No missing values.
6,Missing values,reviews,review_comment_title,87656,88.34% missing; assess in context.
7,Missing values,reviews,review_comment_message,58247,58.70% missing; assess in context.
8,Missing values,products,product_category_name,610,1.85% missing; assess in context.
9,Missing values,products,product_name_lenght,610,1.85% missing; assess in context.


## 11. CallVerse implications

Order, customer, and product identifiers and order milestones can describe observed commerce behavior, subject to missingness and join checks. Approval, handoff, actual delivery, and estimated delivery timestamps need careful treatment; delay features should use comparable timestamps. Validate keys before joins and remember that repeated order IDs in items or payments reflect table granularity. Identify people with `customer_unique_id`, since `customer_id` is created per order. Use (`review_id`, `order_id`) as the review record key, and decide which review counts for orders with several. Measure late delivery by calendar date, because the estimate has no time of day.

Olist is an e-commerce dataset, not a call-center dataset. It contains no actual customer contact events, agent assignments, call durations, queue waiting times, or abandonment records. Contact-demand modeling therefore requires transparent assumptions or a contact-generation mechanism. A delay or cancellation does not prove that a customer contacted support.

## 12. Conclusion and next steps

Phase 1B identifies missingness, duplicate patterns, identifier relationships, timestamp consistency, and order-status observations. It does not perform final cleaning or imputation. The next phase is **Phase 1C — Data Cleaning and Preparation**, with decisions based on the findings documented here.